# 🧠 Python Advanced OOP + File Processing Masterclass

This notebook is designed specifically to make you **actually implement OOP**, not just recognize definitions.

It combines:

- Advanced OOP
- File processing
- Exceptions
- Dictionaries / lists / tuples / sets
- JSON
- CSV
- `pathlib`
- Serialization
- Dataclasses
- Type hints
- Enums
- Composition
- Inheritance
- Polymorphism
- Abstract Base Classes
- `super()` and MRO
- Class / static / instance methods
- Properties
- Encapsulation
- Magic methods
- Operator overloading
- Duck typing
- Custom exceptions
- Context managers
- Integrated lab-style projects

## How to use this notebook

Do **not** read this like a textbook.

For every exercise:

```text
Understand → Predict → Implement → Test → Break it → Fix it
```

The exercises deliberately become harder.

🟢 = foundation  
🟡 = normal lab  
🟠 = advanced  
🔴 = mastery

The final sections require you to combine many concepts without being told exactly which concept to use.

# 0. The OOP mental model

Before syntax, understand this:

```text
CLASS
  ↓
blueprint

OBJECT
  ↓
actual instance of the class

ATTRIBUTE
  ↓
data/state

METHOD
  ↓
behavior

INHERITANCE
  ↓
reuse/extend another class

POLYMORPHISM
  ↓
same interface → different behavior

ABSTRACTION
  ↓
define what must exist without defining every detail

ENCAPSULATION
  ↓
control how state is accessed/changed
```

When designing a class, ask:

1. What does this object **know**?
2. What can this object **do**?
3. What state should be protected?
4. What behavior should subclasses customize?
5. What data needs to be saved to a file?
6. What invalid states should be impossible?

# 1. Classes, objects, `self`

```python
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    def passed(self):
        return self.score >= 50
```

```python
s = Student("Alice", 80)
```

Think:

```text
Student
 ├── name
 ├── score
 └── passed()
```

`self` means **the current object**.

So:

```python
s.passed()
```

causes Python to call:

```python
Student.passed(s)
```

conceptually.

`self.name` belongs to one particular object.

## 🟢 Exercise 1 — Build your first useful class

Create a `BankAccount` class.

Requirements:
- `owner`
- `balance`
- `deposit(amount)`
- `withdraw(amount)`
- `display()`

Rules:
- deposit must be positive
- withdrawal must be positive
- cannot withdraw more than the balance

For now, do not use custom exceptions.
Use `ValueError`.

Test at least:
- normal deposit
- normal withdrawal
- invalid deposit
- invalid withdrawal
- withdrawal larger than balance

In [ ]:
class BankAccount:
    pass

# Test your class here

# 2. Instance variables vs class variables

```python
class Student:
    school = "KU"

    def __init__(self, name):
        self.name = name
```

Here:

```text
Student.school → class variable
student.name   → instance variable
```

The class variable is shared through the class unless an instance shadows it.

### Important

```python
Student.school = "ABC"
```

changes the class attribute.

But:

```python
student.school = "XYZ"
```

creates an instance attribute named `school` for that object.

This distinction is extremely important when debugging OOP.

## 🟡 Exercise 2 — Class state

Create:

```python
class Player:
    player_count = 0
```

Every time a `Player` object is created:
- increase `player_count`

Then:

```python
p1 = Player(...)
p2 = Player(...)
p3 = Player(...)
```

should make:

```python
Player.player_count == 3
```

Explain why `self.player_count += 1` and `Player.player_count += 1` are not always equivalent.

# 3. Instance / class / static methods

## Instance method

```python
def add_score(self, points):
    self.score += points
```

Needs an object.

## Class method

```python
@classmethod
def from_string(cls, text):
    ...
```

Receives the class as `cls`.

Excellent for **alternative constructors / factory methods**.

## Static method

```python
@staticmethod
def valid_score(score):
    return 0 <= score <= 100
```

Doesn't automatically receive `self` or `cls`.

### Mental shortcut

```text
needs object state? → self
needs class state / creates class object? → cls
needs neither? → staticmethod
```

## 🟡 Exercise 3 — Three method types

Build:

```python
class Temperature:
```

Requirements:

### Instance method
```python
to_fahrenheit()
```

### Class method
```python
from_fahrenheit(value)
```

Example:

```python
t = Temperature.from_fahrenheit(212)
```

### Static method
```python
is_valid(value)
```

Return `True` if the temperature is above absolute zero.

Then explain why each method belongs in its category.

# 4. Alternative constructors

This pattern is extremely useful:

```python
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    @classmethod
    def from_string(cls, text):
        name, score = text.split(",")
        return cls(name, int(score))
```

Now:

```python
student = Student.from_string("Alice,85")
```

Why use `cls` instead of writing `Student(...)`?

Because subclasses can inherit the class method and receive the subclass as `cls`.

That preserves polymorphism.

## 🟡 Exercise 4 — Multiple constructors

Create:

```python
class Product:
```

Normal:

```python
Product("P001", "Keyboard", 1200)
```

From CSV-style text:

```python
Product.from_csv("P001,Keyboard,1200")
```

From dictionary:

```python
Product.from_dict({
    "id": "P001",
    "name": "Keyboard",
    "price": 1200
})
```

Implement:

```python
to_dict()
```

too.

You are now building a tiny serialization layer.

# 5. Encapsulation

Python does not have Java-style strict private fields, but it has conventions and name mangling.

```python
self._value
```

means:

> internal/protected-style attribute

```python
self.__value
```

uses name mangling.

Example:

```python
class Account:
    def __init__(self, balance):
        self.__balance = balance

    def get_balance(self):
        return self.__balance
```

A better modern pattern is often a property.

# 6. Properties

```python
class Student:
    def __init__(self, score):
        self._score = 0
        self.score = score

    @property
    def score(self):
        return self._score

    @score.setter
    def score(self, value):
        if not 0 <= value <= 100:
            raise ValueError("Score must be 0..100")
        self._score = value
```

The user writes:

```python
student.score = 90
```

but the setter controls what happens.

This is a major OOP pattern:

```text
public-looking interface
        ↓
validation / controlled state
        ↓
internal data
```

## 🟡 Exercise 5 — Protected state

Create a `Course` class:

- `name`
- `_capacity`
- `_students`

Property:

```python
capacity
```

Rules:
- capacity must be > 0
- cannot reduce capacity below current number of students

Method:

```python
enroll(student)
```

Try deliberately breaking your class with invalid states.

# 7. Inheritance

Basic form:

```python
class Animal:
    def speak(self):
        return "Some sound"

class Dog(Animal):
    def speak(self):
        return "Woof"
```

`Dog` inherits from `Animal`.

A child can:
- inherit methods
- add methods
- add attributes
- override methods

The key relationship is:

```text
Dog IS-A Animal
```

# 8. `super()` — understand this, don't memorize it

```python
class Person:
    def __init__(self, name):
        self.name = name

class Student(Person):
    def __init__(self, name, student_id):
        super().__init__(name)
        self.student_id = student_id
```

`super()` allows the child to continue the inherited behavior.

Important advanced fact:

> `super()` means "continue through the MRO", not simply "call my parent".

This becomes important with multiple inheritance.

## 🟡 Exercise 6 — Inheritance chain

Build:

```text
Person
  ↓
Student
  ↓
GraduateStudent
```

`Person`:
- name

`Student`:
- student_id

`GraduateStudent`:
- thesis_title

Each constructor should use `super()`.

Then override:

```python
describe()
```

so that each level adds information while still using the parent's implementation.

# 9. Method Resolution Order (MRO)

Python searches classes according to an MRO.

```python
class A:
    def show(self):
        print("A")

class B(A):
    def show(self):
        print("B")
        super().show()

class C(B):
    def show(self):
        print("C")
        super().show()
```

```python
C().show()
```

prints:

```text
C
B
A
```

Check:

```python
print(C.mro())
```

Understanding this makes multiple inheritance much less mysterious.

## 🟠 Exercise 7 — MRO prediction

Without running:

```python
class A:
    def hello(self):
        print("A")

class B(A):
    def hello(self):
        print("B")
        super().hello()

class C(A):
    def hello(self):
        print("C")
        super().hello()

class D(B, C):
    def hello(self):
        print("D")
        super().hello()
```

Predict:

```python
D().hello()
```

Then predict:

```python
D.mro()
```

Only then run it.

# 10. Multiple inheritance

Python supports:

```python
class Duck(Flyable, Swimmable):
    pass
```

This can be useful for independent capabilities.

Prefer:

```text
Flyable
Swimmable
```

as capability/mixin-style classes rather than forcing unrelated classes into a fake hierarchy.

Be careful with multiple inheritance because MRO matters.

# 11. Polymorphism

The important idea:

```text
same interface
      ↓
different implementation
```

Example:

```python
class Dog:
    def speak(self):
        return "Woof"

class Cat:
    def speak(self):
        return "Meow"

animals = [Dog(), Cat()]

for animal in animals:
    print(animal.speak())
```

The loop does not need:

```python
if isinstance(animal, Dog):
...
elif isinstance(animal, Cat):
...
```

That is the power of polymorphism.

## 🟡 Exercise 8 — Payment polymorphism

Create:

```text
Payment
├── CreditCardPayment
├── BankTransferPayment
└── CashPayment
```

Each must implement:

```python
pay(amount)
```

Create a list of mixed payment objects and process all of them with one loop.

Then add a new:

```text
CryptoPayment
```

without changing the processing loop.

# 12. Abstract Base Classes

Use:

```python
from abc import ABC, abstractmethod
```

Example:

```python
class Shape(ABC):

    @abstractmethod
    def area(self):
        pass
```

This says:

> Every concrete Shape must provide `area()`.

A subclass:

```python
class Circle(Shape):
    def area(self):
        ...
```

can be instantiated.

But:

```python
Shape()
```

cannot be instantiated while it has unimplemented abstract methods.

## 🟠 Exercise 9 — Abstract data processor

Create:

```text
DataProcessor (ABC)
├── TextProcessor
├── CSVProcessor
└── JSONProcessor
```

Abstract method:

```python
load(filename)
```

Each subclass loads its format differently.

Then write:

```python
def inspect_processor(processor):
    data = processor.load(...)
```

The caller should depend on the interface, not the concrete class.

# 13. Operator overloading / magic methods

Python uses dunder methods for many operators.

Examples:

```text
+      → __add__
-      → __sub__
*      → __mul__
==     → __eq__
<      → __lt__
>      → __gt__
str()  → __str__
len()  → __len__
```

Example:

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Point(
            self.x + other.x,
            self.y + other.y
        )

    def __str__(self):
        return f"Point({self.x}, {self.y})"
```

Then:

```python
p3 = p1 + p2
print(p3)
```

## 🟠 Exercise 10 — Build a custom `Fraction`

Create:

```python
class Fraction:
```

Requirements:
- numerator
- denominator
- reject denominator = 0
- simplify using GCD

Implement:

```python
__add__
__sub__
__mul__
__eq__
__str__
```

Examples:

```python
Fraction(1, 2) + Fraction(1, 3)
Fraction(2, 4) == Fraction(1, 2)
```

This is an excellent exercise because it combines:
- OOP
- validation
- exceptions
- operator overloading
- arithmetic

# 14. Duck typing

Python often cares about behavior rather than inheritance.

```python
def make_it_speak(obj):
    obj.speak()
```

Any object with a compatible `speak()` method can work.

```python
class Dog:
    def speak(self):
        return "Woof"

class Robot:
    def speak(self):
        return "Beep"
```

Neither needs to inherit from the same class.

# 15. Dataclasses

For data-heavy classes:

```python
from dataclasses import dataclass

@dataclass
class Student:
    name: str
    score: float
```

Python generates useful boilerplate such as:
- `__init__`
- `__repr__`
- `__eq__`

You can still add methods.

Frozen:

```python
@dataclass(frozen=True)
class Species:
    name: str
    rarity: str
```

Useful when the object should not be mutated after creation.

## 🟡 Exercise 11 — Dataclass + validation

Create:

```python
@dataclass
class Product:
    product_id: str
    name: str
    price: float
    stock: int
```

Add:

```python
validate()
```

Rules:
- ID cannot be empty
- price >= 0
- stock >= 0

Then implement:

```python
to_dict()
from_dict()
```

Use `@classmethod` for `from_dict`.

# 16. FILE PROCESSING — the mental model

Python file work is fundamentally:

```text
OPEN
 ↓
READ / WRITE
 ↓
CLOSE
```

Basic:

```python
file = open("data.txt", "r")
content = file.read()
file.close()
```

But the preferred pattern is:

```python
with open("data.txt", "r") as file:
    content = file.read()
```

The `with` block handles cleanup.

### Modes

```text
"r"  read
"w"  write / overwrite
"a"  append
"x"  create, fail if exists
"b"  binary mode
"t"  text mode
```

Examples:

```python
"rb"
"wb"
"r+"
```

# 17. Reading files

### Read everything

```python
with open("data.txt", "r") as f:
    content = f.read()
```

### Read one line

```python
with open("data.txt", "r") as f:
    line = f.readline()
```

### Read all lines

```python
with open("data.txt", "r") as f:
    lines = f.readlines()
```

### Iterate

Usually best for large text files:

```python
with open("data.txt", "r") as f:
    for line in f:
        print(line.strip())
```

Remember:

```python
strip()
```

removes surrounding whitespace/newline.

## 🟢 Exercise 12 — Basic text file

Create a file:

```text
students.txt
```

containing:

```text
Alice,85
Bob,72
Carol,91
Dan,65
```

Write code to:
1. open it
2. read it
3. print every line
4. split each line by comma
5. convert score to `int`
6. calculate average score
7. find highest-scoring student

In [ ]:
# Create your test file first
with open("students.txt", "w") as f:
    f.write("Alice,85\n")
    f.write("Bob,72\n")
    f.write("Carol,91\n")
    f.write("Dan,65\n")

# Now read and process it

# 18. `write()` vs `writelines()`

```python
f.write("hello")
```

writes one string.

For multiple strings:

```python
f.writelines([
    "Alice\\n",
    "Bob\\n"
])
```

Important:

`writelines()` does **not** automatically add newlines.

So:

```python
f.writelines(["A", "B"])
```

produces:

```text
AB
```

not:

```text
A
B
```

## 🟢 Exercise 13 — Write and append

Create `log.txt`.

Part A:
- write three log messages

Part B:
- reopen with `"a"`
- append two more messages

Part C:
- read the complete file
- count the number of lines

Part D:
- overwrite the file with `"w"`
- verify that the old content disappeared

Be able to explain exactly what `"w"` and `"a"` do.

# 19. File pointers: `tell()` and `seek()`

A file has a current position.

```python
with open("data.txt", "r") as f:
    print(f.tell())
```

Move the position:

```python
f.seek(0)
```

Example:

```python
with open("data.txt", "r") as f:
    print(f.read(5))
    print(f.tell())

    f.seek(0)
    print(f.read(5))
```

This is especially useful when understanding how file reading works internally.

## 🟡 Exercise 14 — File pointer prediction

Create:

```text
abcdef
```

Then predict:

```python
with open("data.txt") as f:
    print(f.read(2))
    print(f.tell())
    print(f.read(2))
    f.seek(1)
    print(f.read(3))
```

Write the expected output before running.

# 20. `read`, `readline`, `readlines` — know the difference

```text
read()
    ↓
one string containing all requested content

readline()
    ↓
one line

readlines()
    ↓
list of lines
```

For example:

```python
["Alice\n", "Bob\n", "Carol\n"]
```

Then:

```python
for line in lines:
    line = line.strip()
```

is a common processing pattern.

# 21. File exceptions

File processing can fail.

Typical exceptions:

```python
FileNotFoundError
PermissionError
IsADirectoryError
UnicodeDecodeError
```

Example:

```python
try:
    with open("missing.txt") as f:
        data = f.read()
except FileNotFoundError as e:
    print(f"File not found: {e}")
```

Use specific exceptions where possible.

## 🟡 Exercise 15 — Safe file reader

Write:

```python
def read_text(filename):
    ...
```

Requirements:
- return the file contents
- handle `FileNotFoundError`
- handle `PermissionError`
- always use `with`
- do not use a giant bare `except`

Bonus:
- return a useful custom exception instead of leaking low-level errors.

# 22. CSV processing without pandas

Python's standard library provides:

```python
import csv
```

Reading:

```python
with open("students.csv", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        print(row)
```

For:

```text
name,score
Alice,85
Bob,72
```

each row behaves like a dictionary:

```python
{
    "name": "Alice",
    "score": "85"
}
```

Remember: CSV values are strings unless you convert them.

## 🟡 Exercise 16 — CSV → objects

Given:

```text
products.csv

id,name,price,stock
P001,Keyboard,1200,10
P002,Mouse,500,25
P003,Monitor,5000,5
```

Create `Product` objects from every row.

Then:
- calculate total inventory value
- find the most expensive product
- find products with stock < 10
- sort products by price

# 23. Writing CSV

```python
import csv

rows = [
    {"name": "Alice", "score": 85},
    {"name": "Bob", "score": 72}
]

with open("output.csv", "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["name", "score"]
    )

    writer.writeheader()
    writer.writerows(rows)
```

This pattern is extremely useful when your objects have `to_dict()`.

# 24. JSON processing

JSON maps naturally to Python:

```text
JSON object → dict
JSON array  → list
string      → str
number      → int/float
true        → True
false       → False
null        → None
```

Write:

```python
import json

with open("data.json", "w") as f:
    json.dump(data, f, indent=4)
```

Read:

```python
with open("data.json", "r") as f:
    data = json.load(f)
```

## 🟡 Exercise 17 — JSON storage

Create a list of student dictionaries:

```python
students = [
    {"id": "S01", "name": "Alice", "score": 85},
    {"id": "S02", "name": "Bob", "score": 72},
]
```

Write it to:

```text
students.json
```

Then:
1. load it
2. add a new student
3. change Bob's score
4. save it again
5. reload it
6. verify the changes

# 25. `pathlib` — modern file/path handling

Instead of manually manipulating path strings:

```python
from pathlib import Path

path = Path("data") / "students.txt"
```

Useful:

```python
path.exists()
path.is_file()
path.is_dir()
path.name
path.suffix
path.parent
```

Read:

```python
text = path.read_text()
```

Write:

```python
path.write_text("hello")
```

For beginner labs, `open()` is still important to know. `pathlib` is the cleaner tool for path management.

## 🟡 Exercise 18 — File manager

Write a function:

```python
def inspect_file(path):
```

that reports:
- exists?
- file or directory?
- filename
- suffix
- parent directory
- size in bytes

Use `pathlib`.

# 26. Serialization — connecting OOP to files

Objects cannot always be directly written to JSON.

For example:

```python
student = Student("Alice", 85)
```

JSON doesn't know what a `Student` object is.

So create a representation:

```python
student.to_dict()
```

→

```python
{
    "name": "Alice",
    "score": 85
}
```

Then:

```python
Student.from_dict(data)
```

reconstructs the object.

This pattern is:

```text
OBJECT
  ↓
to_dict()
  ↓
dict
  ↓
json.dump()
  ↓
JSON file
  ↓
json.load()
  ↓
dict
  ↓
from_dict()
  ↓
OBJECT
```

## 🟠 Exercise 19 — Full object persistence

Create:

```python
@dataclass
class Player:
    user_id: str
    name: str
    score: int
```

Implement:

```python
to_dict()
from_dict()
```

Then functions:

```python
save_players(players, filename)
load_players(filename)
```

Requirements:
- save multiple objects
- load them back into actual `Player` objects
- handle missing file
- handle malformed JSON
- validate score

# 27. Custom exceptions + OOP

A custom exception is a class.

```python
class AppError(Exception):
    pass
```

You can create a hierarchy:

```text
AppError
├── ValidationError
├── NotFoundError
└── InvalidStateError
```

This is useful because callers can catch either:

```python
except ValidationError:
```

or:

```python
except AppError:
```

The lecture also demonstrates storing custom data inside exceptions and calling:

```python
super().__init__(message)
```

so the parent `Exception` receives the message.

## 🟠 Exercise 20 — Build an application error system

Create:

```python
class AppError(Exception):
    def __init__(self, message, field=None):
        ...
```

Then:

```python
class ValidationError(AppError):
    pass

class NotFoundError(AppError):
    pass

class InvalidStateError(AppError):
    pass
```

Use them inside a small `StudentRepository`.

Examples:
- invalid score → `ValidationError`
- missing student → `NotFoundError`
- deleting while locked → `InvalidStateError`

# 28. Context managers — advanced file/OOP connection

The `with` statement works because objects can implement:

```python
__enter__()
__exit__()
```

Conceptually:

```python
with resource as x:
    ...
```

calls something like:

```python
x = resource.__enter__()

try:
    ...
finally:
    resource.__exit__(...)
```

This is why `with open(...)` can automatically clean up the file.

## 🟠 Exercise 21 — Build your own context manager

Create:

```python
class FileLogger:
```

Usage:

```python
with FileLogger("app.log") as logger:
    logger.write("Started")
    logger.write("Doing work")
```

Requirements:
- open the file in `__enter__`
- return an object from `__enter__`
- write messages
- close the file in `__exit__`
- prove that the file closes even if an exception occurs

Bonus:
- write `"ERROR"` to the file if an exception occurred

# 29. Composition

Inheritance:

```text
Dog IS-A Animal
```

Composition:

```text
Car HAS-A Engine
```

Example:

```python
class Engine:
    def start(self):
        return "started"

class Car:
    def __init__(self):
        self.engine = Engine()
```

Composition is often preferable when objects are built from other objects.

## 🟠 Exercise 22 — Composition

Build:

```text
Computer
├── CPU
├── Memory
└── Storage
```

`Computer` should contain objects representing each component.

Implement:

```python
computer.boot()
computer.specs()
```

Do not use inheritance for CPU/Memory/Storage.

# 30. Repository pattern — extremely useful practice

A repository separates storage logic from business logic.

Example idea:

```python
class StudentRepository:
    def __init__(self, filename):
        self.filename = filename

    def save(self, students):
        ...

    def load(self):
        ...

    def find_by_id(self, student_id):
        ...
```

Then another class can use it:

```python
class StudentService:
    def __init__(self, repository):
        self.repository = repository
```

This is composition + separation of responsibilities.

## 🟠 Exercise 23 — JSON repository

Implement:

```text
Student
StudentRepository
StudentService
```

### Student
Stores:
- id
- name
- score

### Repository
Responsible ONLY for:
- loading
- saving
- finding

### Service
Responsible for:
- adding students
- validating rules
- updating scores
- calculating statistics

Use dependency injection:

```python
service = StudentService(repository)
```

Do not let `Student` itself open files.

# 31. Advanced file-processing challenge

## 🔴 Exercise 24 — Log analyzer

File:

```text
app.log
```

contains:

```text
INFO,login,user01
INFO,login,user02
ERROR,db,timeout
INFO,logout,user01
WARNING,auth,3 failed attempts
ERROR,db,connection lost
```

Build a program that:
1. reads the file
2. parses every line
3. creates a `LogEntry` dataclass
4. validates the severity
5. stores entries
6. counts each severity
7. finds all errors
8. finds all events related to `db`
9. writes an `error_report.txt`
10. writes a JSON copy
11. handles malformed lines using a custom exception

Use:
- file processing
- dataclass
- custom exception
- list/dict
- JSON
- sorting
- `with`

# 32. Advanced OOP challenge — RPG system

## 🔴 Exercise 25

Design:

```text
Character (ABC)
├── Warrior
├── Mage
└── Archer
```

Each has:
- name
- HP
- attack power

Abstract:

```python
attack(target)
```

Requirements:
- polymorphic attacks
- validation with properties
- custom exceptions
- `__str__`
- equality based on character ID
- inventory using composition
- save/load characters as JSON
- class method `from_dict`
- static method for validating IDs
- use inheritance where appropriate
- use composition for inventory

Then create:

```python
characters = [Warrior(...), Mage(...), Archer(...)]
```

and simulate a battle loop using polymorphism.

# 33. Advanced OOP challenge — study/session system

This is intentionally close to the type of architecture used in larger projects.

Design:

```text
AppError
├── ValidationError
├── NotFoundError
└── InvalidStateError

Player
StudySession
SessionRepository
SessionService

SessionStatus (Enum)
```

### Player
- ID
- name
- score

### StudySession
- session ID
- player
- task
- elapsed seconds
- status

### SessionRepository
- save
- load
- find

### SessionService
- start session
- stop session
- add time
- calculate summary

Rules:
- cannot stop an unstarted session
- cannot add negative time
- cannot add time to a completed session
- player score cannot be negative
- missing session → `NotFoundError`

Persist everything to JSON.

This exercise combines:

```text
OOP
+ dataclass
+ Enum
+ custom exceptions
+ properties
+ composition
+ file handling
+ JSON
+ dictionaries/lists
+ class methods
```

# 34. FINAL MASTER CHALLENGE — build a complete application

## 🔴 Student Management System

You must design the architecture yourself.

### Requirements

The application manages:

```text
Student
Course
Enrollment
StudentRepository
CourseRepository
StudentService
```

### Student
- student ID
- name
- email
- courses

### Course
- course code
- name
- capacity
- enrolled students

### Enrollment
- student
- course
- grade

### Rules

- IDs cannot be empty
- email must contain `@`
- course capacity must be positive
- cannot enroll the same student twice
- cannot exceed capacity
- grade must be 0–100
- cannot assign a grade before enrollment

### Persistence

Use:

```text
students.json
courses.json
enrollments.json
```

### OOP requirements

You MUST use:
- dataclasses
- properties
- class methods
- static methods
- composition
- custom exceptions
- Enum
- at least one abstract class
- polymorphism
- `__str__`
- `__eq__`

### File requirements

You MUST demonstrate:
- `"r"`
- `"w"`
- `"a"`
- `with open(...)`
- JSON load/save
- `pathlib`
- exception handling

### Program features

Implement:

```text
1. Add student
2. Add course
3. Enroll student
4. Assign grade
5. Search student
6. Search course
7. Show student report
8. Show course report
9. Save
10. Load
```

### Important

Do NOT put everything in one class.

Your goal is to demonstrate **separation of responsibility**.

# 35. Code-reading mastery

For each code fragment below, explain the OOP concept before running it.

### A

```python
class A:
    x = 10

a = A()
a.x = 20

print(A.x)
print(a.x)
```

### B

```python
class A:
    @classmethod
    def create(cls):
        return cls()
```

Why use `cls`?

### C

```python
class A:
    def save(self):
        ...

class B(A):
    def save(self):
        ...
```

What concept?

### D

```python
class A(ABC):
    @abstractmethod
    def run(self):
        pass
```

What restriction does this create?

### E

```python
with open("x.txt") as f:
    ...
```

What Python protocol makes this work?

### F

```python
obj.to_dict()
json.dump(...)
```

What problem are these two steps solving?

# 36. Debugging OOP — find the bug

## 🟠 Exercise 26

What is wrong?

```python
class Student:
    students = []

    def __init__(self, name):
        self.name = name
        self.students.append(name)

a = Student("Alice")
b = Student("Bob")

print(a.students)
print(b.students)
```

Is the shared list intentional?

Fix it if the requirement is:

> Every Student should have their own list of friends.

## 🟠 Exercise 27 — Inheritance bug

Find the problem:

```python
class Person:
    def __init__(self, name):
        self.name = name

class Student(Person):
    def __init__(self, name, student_id):
        self.student_id = student_id
```

Why does:

```python
Student("Alice", "S01").name
```

fail?

Fix it using `super()`.

## 🟠 Exercise 28 — File bug

Find all problems:

```python
f = open("data.txt", "r")
data = f.read()

if data:
    print(data)

f.close()

print(f.read())
```

Questions:
1. What happens?
2. Why?
3. What is the safer structure?

# 37. Output prediction — file processing

Predict exactly:

```python
with open("demo.txt", "w") as f:
    f.write("ABC")
    f.write("DEF")

with open("demo.txt", "a") as f:
    f.write("GHI")

with open("demo.txt", "r") as f:
    print(f.read())
```

Then change the second mode from `"a"` to `"w"`.

Predict again.

# 38. Output prediction — OOP

Predict:

```python
class A:
    value = 1

    def __init__(self):
        self.value = 2

class B(A):
    value = 3

b = B()

print(A.value)
print(B.value)
print(b.value)
```

Then add:

```python
b.value = 99
```

What changes?

# 39. Master checklist

Before saying "I understand OOP", you should be able to implement these **without looking up syntax**:

## Classes
- [ ] class
- [ ] object
- [ ] `__init__`
- [ ] `self`
- [ ] instance variables
- [ ] class variables
- [ ] instance methods
- [ ] class methods
- [ ] static methods

## OOP design
- [ ] encapsulation
- [ ] inheritance
- [ ] overriding
- [ ] `super()`
- [ ] MRO
- [ ] multiple inheritance
- [ ] polymorphism
- [ ] duck typing
- [ ] abstraction
- [ ] ABC
- [ ] composition
- [ ] properties
- [ ] dataclasses
- [ ] enums
- [ ] magic methods

## Persistence
- [ ] `open`
- [ ] `close`
- [ ] `with open`
- [ ] `r`
- [ ] `w`
- [ ] `a`
- [ ] `x`
- [ ] `read`
- [ ] `readline`
- [ ] `readlines`
- [ ] iteration
- [ ] `write`
- [ ] `writelines`
- [ ] `seek`
- [ ] `tell`
- [ ] CSV
- [ ] JSON
- [ ] pathlib

## Exceptions
- [ ] `try`
- [ ] specific `except`
- [ ] `as e`
- [ ] `else`
- [ ] `finally`
- [ ] `raise`
- [ ] `assert`
- [ ] custom exceptions
- [ ] exception inheritance
- [ ] `super().__init__`

## Integration
- [ ] object → dict
- [ ] dict → object
- [ ] object → JSON
- [ ] JSON → object
- [ ] repository
- [ ] service
- [ ] separation of responsibility
- [ ] validation
- [ ] persistence

# 40. The ultimate OOP problem-solving framework

When you receive a difficult lab problem, do this:

## Step 1 — Identify nouns

Example:

> A university stores students, courses, and enrollments.

Nouns:

```text
Student
Course
Enrollment
```

These are candidates for classes.

## Step 2 — Identify data

Ask:

```text
What does each object know?
```

## Step 3 — Identify verbs

Example:

```text
enroll
withdraw
save
load
calculate
```

These are candidates for methods.

## Step 4 — Identify relationships

```text
Student HAS-A Enrollment
Course HAS-A Enrollment

Dog IS-A Animal
```

Use composition for HAS-A and inheritance for genuine IS-A relationships.

## Step 5 — Identify invalid states

```text
negative score
duplicate enrollment
missing file
invalid ID
```

These become validation + exceptions.

## Step 6 — Identify persistence

Ask:

```text
What must survive after the program closes?
```

Then design:

```text
object
 ↓
to_dict
 ↓
JSON
```

and:

```text
JSON
 ↓
dict
 ↓
from_dict
 ↓
object
```

## Step 7 — Identify polymorphism

Ask:

> Do several objects need to perform the same operation differently?

If yes, consider:

```text
ABC
inheritance
overriding
polymorphism
```

## Step 8 — Separate responsibilities

Avoid:

```python
Student.save_everything_and_validate_and_print_and_load()
```

Prefer:

```text
Student
Repository
Service
```

## Step 9 — Test failure cases

A program isn't finished because the happy path works.

Test:

```text
empty input
wrong type
negative number
missing file
malformed JSON
duplicate data
invalid state
boundary values
```

## Step 10 — Explain your architecture

If you can explain:

> "Why is this a class? Why is this inheritance? Why is this composition? Why is this exception here? Why does this repository handle the file?"

then you actually understand the OOP design.

# 🏁 Final rule

Do not measure your OOP skill by whether you can copy this:

```python
class Student:
    ...
```

Measure it by whether you can receive a problem you've never seen before and independently decide:

```text
What objects exist?
        ↓
What state do they hold?
        ↓
What behavior do they have?
        ↓
How are they related?
        ↓
What should be abstract?
        ↓
What should be polymorphic?
        ↓
What should be composed?
        ↓
What can go wrong?
        ↓
What exceptions represent those failures?
        ↓
What needs to be saved?
        ↓
How do objects become dictionaries/JSON?
        ↓
How do we load them back?
```

That is the skill this notebook is designed to train.